# Chapter 10: Combining Datasets: Merge, Join, and Concatenate

Companion notebook for *Master Data Cleaning: A Hands-On Beginner's Guide with Python, pandas & NumPy*.

Run the cells from top to bottom (**Kernel → Restart Kernel and Run All Cells**). The data files this chapter uses are in the same folder as this notebook.

## 10.1 Stacking DataFrames with concat

### `pd.concat()`

In [1]:
import pandas as pd
import numpy as np

q1_sales = pd.DataFrame({
    'date': pd.date_range('2024-01-31', periods=3, freq='ME'),
    'region': ['North', 'North', 'North'],
    'sales': [45000, 52000, 48000],
})
q2_sales = pd.DataFrame({
    'date': pd.date_range('2024-04-30', periods=3, freq='ME'),
    'region': ['North', 'North', 'North'],
    'sales': [61000, 57000, 63000],
})
q1_sales

,date,region,sales
0,2024-01-31,North,45000
1,2024-02-29,North,52000
2,2024-03-31,North,48000


In [2]:
q2_sales

,date,region,sales
0,2024-04-30,North,61000
1,2024-05-31,North,57000
2,2024-06-30,North,63000


In [3]:
half_year = pd.concat([q1_sales, q2_sales])
half_year

,date,region,sales
0,2024-01-31,North,45000
1,2024-02-29,North,52000
2,2024-03-31,North,48000
0,2024-04-30,North,61000
1,2024-05-31,North,57000
2,2024-06-30,North,63000


### Resetting the index with `ignore_index=True`

In [4]:
half_year = pd.concat([q1_sales, q2_sales], ignore_index=True)
half_year

,date,region,sales
0,2024-01-31,North,45000
1,2024-02-29,North,52000
2,2024-03-31,North,48000
3,2024-04-30,North,61000
4,2024-05-31,North,57000
5,2024-06-30,North,63000


In [5]:
q3_sales = pd.DataFrame({
    'date': pd.date_range('2024-07-31', periods=3, freq='ME'),
    'region': ['North', 'North', 'North'],
    'sales': [68000, 72000, 70000],
})
q4_sales = pd.DataFrame({
    'date': pd.date_range('2024-10-31', periods=3, freq='ME'),
    'region': ['North', 'North', 'North'],
    'sales': [75000, 82000, 79000],
})
full_year = pd.concat(
    [q1_sales, q2_sales, q3_sales, q4_sales], ignore_index=True
)
full_year

,date,region,sales
0,2024-01-31,North,45000
1,2024-02-29,North,52000
2,2024-03-31,North,48000
3,2024-04-30,North,61000
4,2024-05-31,North,57000
5,2024-06-30,North,63000
6,2024-07-31,North,68000
7,2024-08-31,North,72000
8,2024-09-30,North,70000
9,2024-10-31,North,75000


### Mismatched columns

In [6]:
north = pd.DataFrame({
    'month': ['Jan', 'Feb', 'Mar'],
    'region': ['North', 'North', 'North'],
    'sales': [45000, 52000, 48000],
    'units': [450, 520, 480],
})
south = pd.DataFrame({
    'month': ['Jan', 'Feb', 'Mar'],
    'region': ['South', 'South', 'South'],
    'sales': [38000, 41000, 39000],
    'customers': [150, 165, 158],
})
pd.concat([north, south], ignore_index=True)

,month,region,sales,units,customers
0,Jan,North,45000,450.0,NaN
1,Feb,North,52000,520.0,NaN
2,Mar,North,48000,480.0,NaN
3,Jan,South,38000,NaN,150.0
4,Feb,South,41000,NaN,165.0
5,Mar,South,39000,NaN,158.0


In [7]:
pd.concat([north, south], join='inner', ignore_index=True)

,month,region,sales
0,Jan,North,45000
1,Feb,North,52000
2,Mar,North,48000
3,Jan,South,38000
4,Feb,South,41000
5,Mar,South,39000


### Labeling the source with `keys=`

In [8]:
by_quarter = pd.concat(
    [q1_sales, q2_sales],
    keys=['Q1', 'Q2'],
    names=['quarter', 'row'],
)
by_quarter

date region  sales
quarter row                         
Q1      0   2024-01-31  North  45000
        1   2024-02-29  North  52000
        2   2024-03-31  North  48000
Q2      0   2024-04-30  North  61000
        1   2024-05-31  North  57000
        2   2024-06-30  North  63000

In [9]:
by_quarter.loc['Q2']

,date,region,sales
row,,,
0,2024-04-30,North,61000
1,2024-05-31,North,57000
2,2024-06-30,North,63000


### Placing DataFrames side by side with `axis=1`

In [10]:
monthly_sales = pd.DataFrame({
    'month': ['Jan', 'Feb', 'Mar'],
    'sales': [45000, 52000, 48000],
})
monthly_units = pd.DataFrame({
    'units': [450, 520, 480],
    'returns': [12, 15, 11],
})
pd.concat([monthly_sales, monthly_units], axis=1)

,month,sales,units,returns
0,Jan,45000,450,12
1,Feb,52000,520,15
2,Mar,48000,480,11


## 10.2 Merging DataFrames on a Key

### `pd.merge()`

In [11]:
purchases = pd.DataFrame({
    'customer_id': [101, 102, 103, 104],
    'amount': [250, 150, 300, 200],
})
contacts = pd.DataFrame({
    'customer_id': [101, 102, 103, 105],
    'email': ['alice@example.com', 'bob@example.com',
              'carol@example.com', 'dave@example.com'],
})
pd.merge(purchases, contacts, on='customer_id')

,customer_id,amount,email
0,101,250,alice@example.com
1,102,150,bob@example.com
2,103,300,carol@example.com


### Overlapping column names and `suffixes=`

In [12]:
sales_2023 = pd.DataFrame({
    'product_id': ['A101', 'A102', 'A103'],
    'revenue': [50000, 75000, 60000],
})
sales_2024 = pd.DataFrame({
    'product_id': ['A101', 'A102', 'A104'],
    'revenue': [55000, 80000, 45000],
})
pd.merge(sales_2023, sales_2024, on='product_id')

,product_id,revenue_x,revenue_y
0,A101,50000,55000
1,A102,75000,80000


In [13]:
pd.merge(sales_2023, sales_2024, on='product_id',
         suffixes=('_2023', '_2024'))

,product_id,revenue_2023,revenue_2024
0,A101,50000,55000
1,A102,75000,80000


### One-to-one, one-to-many and many-to-many

In [14]:
orders = pd.DataFrame({
    'customer_id': [1, 1, 2],
    'order_id': ['A', 'B', 'C'],
    'amount': [100, 150, 200],
})
locations = pd.DataFrame({
    'customer_id': [1, 2],
    'city': ['New York', 'Philadelphia'],
})
pd.merge(orders, locations, on='customer_id')

,customer_id,order_id,amount,city
0,1,A,100,New York
1,1,B,150,New York
2,2,C,200,Philadelphia


In [15]:
enrollments = pd.DataFrame({
    'course': ['Math', 'Math', 'Physics'],
    'student': ['Aisha', 'Ben', 'Chen'],
})
schedules = pd.DataFrame({
    'course': ['Math', 'Math'],
    'time': ['9am', '2pm'],
})
pd.merge(enrollments, schedules, on='course')

,course,student,time
0,Math,Aisha,9am
1,Math,Aisha,2pm
2,Math,Ben,9am
3,Math,Ben,2pm


### Checking keys with `validate=`

The next cell raises an error on purpose.

In [16]:
pd.merge(orders, locations, on='customer_id',
         validate='one_to_one')

MergeError: Merge keys are not unique in left dataset; not a one-to-one merge
Duplicates in left:
  customer_id
           1 ...

In [17]:
pd.merge(orders, locations, on='customer_id',
         validate='many_to_one')

,customer_id,order_id,amount,city
0,1,A,100,New York
1,1,B,150,New York
2,2,C,200,Philadelphia


## 10.3 Join Types: Inner, Left, Right, and Outer

In [18]:
customers = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
customers

,customer_id,name
0,101,Alice
1,102,Bob
2,103,Carol
3,104,David


In [19]:
orders

,customer_id,order_total
0,101,250.50
1,103,180.00
2,105,420.75


### Inner join

In [20]:
pd.merge(customers, orders, on='customer_id', how='inner')

,customer_id,name,order_total
0,101,Alice,250.5
1,103,Carol,180.0


### Left join

In [21]:
pd.merge(customers, orders, on='customer_id', how='left')

,customer_id,name,order_total
0,101,Alice,250.5
1,102,Bob,NaN
2,103,Carol,180.0
3,104,David,NaN


### Right join

In [22]:
pd.merge(customers, orders, on='customer_id', how='right')

,customer_id,name,order_total
0,101,Alice,250.50
1,103,Carol,180.00
2,105,NaN,420.75


### Outer join

In [23]:
pd.merge(customers, orders, on='customer_id', how='outer')

,customer_id,name,order_total
0,101,Alice,250.50
1,102,Bob,NaN
2,103,Carol,180.00
3,104,David,NaN
4,105,NaN,420.75


### Comparing row counts

In [24]:
for join_type in ['inner', 'left', 'right', 'outer']:
    merged = pd.merge(customers, orders, on='customer_id',
                      how=join_type)
    print(f'{join_type:>5}: {len(merged)} rows')

inner: 2 rows
 left: 4 rows
right: 3 rows
outer: 5 rows


### Tracing rows with `indicator=True`

In [25]:
audit = pd.merge(customers, orders, on='customer_id',
                 how='outer', indicator=True)
audit

,customer_id,name,order_total,_merge
0,101,Alice,250.50,both
1,102,Bob,NaN,left_only
2,103,Carol,180.00,both
3,104,David,NaN,left_only
4,105,NaN,420.75,right_only


In [26]:
audit[audit['_merge'] == 'left_only']

,customer_id,name,order_total,_merge
1,102,Bob,NaN,left_only
3,104,David,NaN,left_only


In [27]:
audit[audit['_merge'] != 'both']

,customer_id,name,order_total,_merge
1,102,Bob,NaN,left_only
3,104,David,NaN,left_only
4,105,NaN,420.75,right_only


In [28]:
audit['_merge'].value_counts()

_merge
left_only     2
both          2
right_only    1
Name: count, dtype: int64

## 10.4 Choosing Keys: Columns, Indexes, and join()

### Different column names with `left_on=` and `right_on=`

In [29]:
shop_orders = pd.DataFrame({
    'order_id': ['O1', 'O2', 'O3'],
    'customer_code': ['C101', 'C102', 'C103'],
    'amount': [250.00, 175.50, 420.75],
})
clients = pd.DataFrame({
    'client_id': ['C101', 'C102', 'C104'],
    'client_name': ['Alpha Sdn Bhd', 'Beta Trading', 'Gamma Ltd'],
})
pd.merge(shop_orders, clients,
         left_on='customer_code', right_on='client_id',
         how='left')

,order_id,customer_code,amount,client_id,client_name
0,O1,C101,250.00,C101,Alpha Sdn Bhd
1,O2,C102,175.50,C102,Beta Trading
2,O3,C103,420.75,NaN,NaN


In [30]:
order_details = pd.merge(
    shop_orders, clients,
    left_on='customer_code', right_on='client_id', how='left',
).drop(columns='client_id')
order_details

,order_id,customer_code,amount,client_name
0,O1,C101,250.00,Alpha Sdn Bhd
1,O2,C102,175.50,Beta Trading
2,O3,C103,420.75,NaN


### Merging on several columns

In [31]:
store_revenue = pd.DataFrame({
    'store_id': ['S1', 'S1', 'S2', 'S2'],
    'date': ['2024-01-01', '2024-01-02', '2024-01-01', '2024-01-02'],
    'revenue': [1000, 1200, 800, 950],
})
store_traffic = pd.DataFrame({
    'store_id': ['S1', 'S1', 'S2'],
    'date': ['2024-01-01', '2024-01-02', '2024-01-01'],
    'visitors': [250, 300, 180],
})
pd.merge(store_revenue, store_traffic, on=['store_id', 'date'])

,store_id,date,revenue,visitors
0,S1,2024-01-01,1000,250
1,S1,2024-01-02,1200,300
2,S2,2024-01-01,800,180


### Merging on the index with `left_index=` and `right_index=`

In [32]:
prices = pd.DataFrame(
    {'price': [299.99, 49.99, 89.99]},
    index=pd.Index(['LAPTOP', 'MOUSE', 'KEYBOARD'], name='product'),
)
stock = pd.DataFrame(
    {'in_stock': [15, 42, 8]},
    index=pd.Index(['LAPTOP', 'MOUSE', 'MONITOR'], name='product'),
)
pd.merge(prices, stock, left_index=True, right_index=True)

,price,in_stock
product,,
LAPTOP,299.99,15
MOUSE,49.99,42


In [33]:
budgets = pd.DataFrame(
    {'budget': [500000, 750000, 300000]},
    index=pd.Index(['SALES', 'ENG', 'HR'], name='dept'),
)
staff = pd.DataFrame({
    'emp_id': [101, 102, 103, 104],
    'name': ['Aisha', 'Ben', 'Chen', 'Devi'],
    'department': ['SALES', 'ENG', 'ENG', 'MARKETING'],
})
pd.merge(staff, budgets, left_on='department', right_index=True)

,emp_id,name,department,budget
0,101,Aisha,SALES,500000
1,102,Ben,ENG,750000
2,103,Chen,ENG,750000


### `df.join()`

In [34]:
employees = pd.DataFrame(
    {'name': ['Aisha', 'Ben', 'Chen']},
    index=pd.Index([101, 102, 103], name='emp_id'),
)
salaries = pd.DataFrame(
    {'salary': [60000, 75000]},
    index=pd.Index([101, 102], name='emp_id'),
)
employees.join(salaries)

,name,salary
emp_id,,
101,Aisha,60000.0
102,Ben,75000.0
103,Chen,NaN


In [35]:
employees.join(salaries, how='inner')

,name,salary
emp_id,,
101,Aisha,60000
102,Ben,75000


In [36]:
regions = pd.DataFrame(
    {'region': ['East', 'West', 'South']},
    index=pd.Index(['C101', 'C102', 'C104'], name='customer_code'),
)
shop_orders.join(regions, on='customer_code')

,order_id,customer_code,amount,region
0,O1,C101,250.00,East
1,O2,C102,175.50,West
2,O3,C103,420.75,NaN


In [37]:
revenue_jan = pd.DataFrame(
    {'revenue': [1000, 1500, 1200]},
    index=['Store A', 'Store B', 'Store C'],
)
revenue_feb = pd.DataFrame(
    {'revenue': [1100, 1600, 1300]},
    index=['Store A', 'Store B', 'Store D'],
)
revenue_jan.join(revenue_feb, lsuffix='_jan', rsuffix='_feb')

,revenue_jan,revenue_feb
Store A,1000,1100.0
Store B,1500,1600.0
Store C,1200,NaN


## 10.5 Patching Gaps with combine_first()

### `df.combine_first()`

In [38]:
catalog = pd.DataFrame({
    'price': [10.00, np.nan, 15.00],
    'stock': [100, 50, np.nan],
}, index=['Pen', 'Ruler', 'Stapler'])
supplier = pd.DataFrame({
    'price': [9.00, 12.00, 14.00],
    'stock': [120, 60, 75],
}, index=['Pen', 'Ruler', 'Stapler'])
catalog.combine_first(supplier)

,price,stock
Pen,10.0,100.0
Ruler,12.0,50.0
Stapler,15.0,75.0


In [39]:
q1_report = pd.DataFrame({
    'revenue': [50000, np.nan, 45000],
    'expenses': [30000, 25000, np.nan],
}, index=['Jan', 'Feb', 'Mar'])
corrections = pd.DataFrame({
    'revenue': [51000, 48000, 44000, 52000],
    'profit': [20000, 23000, 19000, 27000],
}, index=['Jan', 'Feb', 'Mar', 'Apr'])
q1_report.combine_first(corrections)

,revenue,expenses,profit
Apr,52000.0,NaN,27000
Feb,48000.0,25000.0,23000
Jan,50000.0,30000.0,20000
Mar,45000.0,NaN,19000


### Chaining fallbacks

In [40]:
manual = pd.DataFrame({'reading': [100, np.nan, np.nan]},
                      index=['A', 'B', 'C'])
system = pd.DataFrame({'reading': [110, 200, np.nan]},
                      index=['A', 'B', 'C'])
archive = pd.DataFrame({'reading': [120, 210, 300]},
                       index=['A', 'B', 'C'])
manual.combine_first(system).combine_first(archive)

,reading
A,100.0
B,200.0
C,300.0


## 10.6 Automatic Alignment in Arithmetic

In [41]:
store_a = pd.DataFrame({
    'Jan': [100, 150],
    'Feb': [120, 160],
    'Mar': [110, 140],
}, index=['Electronics', 'Clothing'])
store_b = pd.DataFrame({
    'Feb': [90, 130],
    'Mar': [95, 145],
    'Apr': [100, 150],
}, index=['Clothing', 'Home'])
store_a

,Jan,Feb,Mar
Electronics,100,120,110
Clothing,150,160,140


In [42]:
store_b

,Feb,Mar,Apr
Clothing,90,95,100
Home,130,145,150


### Adding with `+`

In [43]:
store_a + store_b

,Apr,Feb,Jan,Mar
Clothing,NaN,250.0,NaN,235.0
Electronics,NaN,NaN,NaN,NaN
Home,NaN,NaN,NaN,NaN


In [44]:
store_a * 2

,Jan,Feb,Mar
Electronics,200,240,220
Clothing,300,320,280


### Filling gaps with `fill_value=`

In [45]:
store_a.add(store_b, fill_value=0)

,Apr,Feb,Jan,Mar
Clothing,100.0,250.0,150.0,235.0
Electronics,NaN,120.0,100.0,110.0
Home,150.0,130.0,NaN,145.0
